In [0]:
%sql

-- Silver — Refined Layer a partir da Bronze:
-- Tipagem correta de colunas (inteiros, datas)
-- Deduplicação (mesma linha pode chegar mais de uma vez em cargas reprocessadas)
-- Mascaramento do `NR_CNPJ` (dado sensível, ainda que de pessoa jurídica)
-- Delta table particionada por competência (`ID_CMPT_MOVEL`), formato final tabular para consumo

In [0]:
%sql
-- utilizando o catalogo e schema padrao
USE CATALOG bmg_saude_desafio;
USE SCHEMA silver;

In [0]:
%sql
--criando a tabela beneficiarios na silver
CREATE TABLE IF NOT EXISTS silver.beneficiarios_ans (
  competencia                 STRING,
  cd_operadora                 STRING,
  nm_razao_social               STRING,
  nr_cnpj_mascarado            STRING,
  modalidade_operadora          STRING,
  sg_uf                        STRING,
  cd_municipio                  STRING,
  nm_municipio                  STRING,
  tp_sexo                      STRING,
  de_faixa_etaria               STRING,
  de_faixa_etaria_reajuste       STRING,
  cd_plano                     STRING,
  tp_vigencia_plano             STRING,
  de_contratacao_plano          STRING,
  de_segmentacao_plano          STRING,
  de_abrangencia_geografica_plano STRING,
  cobertura_assistencial_plano   STRING,
  tipo_vinculo                  STRING,
  qt_beneficiario_ativo          BIGINT,
  qt_beneficiario_aderido        BIGINT,
  qt_beneficiario_cancelado      BIGINT,
  dt_carga                     DATE,
  _ingested_at                  TIMESTAMP
)
USING DELTA
PARTITIONED BY (competencia)
COMMENT 'Silver - dados de beneficiarios ANS tipados, deduplicados e mascarados';

In [0]:
%sql
-- MERGE incremental a partir da Bronze
-- Idempotente: roda todo dia, só insere linhas novas da Bronze que ainda
-- não existem na Silver (chave de negócio: competência + operadora + município + plano + faixa etária + sexo + vínculo).
 
CREATE OR REPLACE TEMPORARY VIEW stg_silver AS
SELECT
  ID_CMPT_MOVEL                                              AS competencia,
  CD_OPERADORA                                                AS cd_operadora,
  TRIM(NM_RAZAO_SOCIAL)                                       AS nm_razao_social,
  CONCAT(LEFT(NR_CNPJ, 2), REPEAT('*', 8), RIGHT(NR_CNPJ, 4)) AS nr_cnpj_mascarado,
  MODALIDADE_OPERADORA                                        AS modalidade_operadora,
  SG_UF                                                       AS sg_uf,
  CD_MUNICIPIO                                                 AS cd_municipio,
  NM_MUNICIPIO                                                 AS nm_municipio,
  TP_SEXO                                                     AS tp_sexo,
  DE_FAIXA_ETARIA                                              AS de_faixa_etaria,
  DE_FAIXA_ETARIA_REAJ                                         AS de_faixa_etaria_reajuste,
  CD_PLANO                                                    AS cd_plano,
  TP_VIGENCIA_PLANO                                            AS tp_vigencia_plano,
  DE_CONTRATACAO_PLANO                                         AS de_contratacao_plano,
  DE_SEGMENTACAO_PLANO                                         AS de_segmentacao_plano,
  DE_ABRG_GEOGRAFICA_PLANO                                     AS de_abrangencia_geografica_plano,
  COBERTURA_ASSIST_PLAN                                        AS cobertura_assistencial_plano,
  TIPO_VINCULO                                                 AS tipo_vinculo,
  TRY_CAST(QT_BENEFICIARIO_ATIVO     AS BIGINT)                AS qt_beneficiario_ativo,
  TRY_CAST(QT_BENEFICIARIO_ADERIDO   AS BIGINT)                AS qt_beneficiario_aderido,
  TRY_CAST(QT_BENEFICIARIO_CANCELADO AS BIGINT)                AS qt_beneficiario_cancelado,
  TRY_CAST(DT_CARGA AS DATE)                                   AS dt_carga,
  _ingested_at
FROM bronze.beneficiarios_ans_raw
QUALIFY ROW_NUMBER() OVER (
  -- chave completa: esse arquivo já vem agregado por combinação de
  -- categoria, então a "linha única" só existe quando olhamos TODAS as
  -- colunas de dimensão junto (faixa etária de reajuste, segmentação,
  -- abrangência, cobertura e vigência do plano também diferenciam
  -- registros - não só operadora/município/plano/faixa/sexo/vínculo)
  PARTITION BY ID_CMPT_MOVEL, CD_OPERADORA, NM_MUNICIPIO, CD_PLANO,
               TP_VIGENCIA_PLANO, DE_CONTRATACAO_PLANO, DE_SEGMENTACAO_PLANO,
               DE_ABRG_GEOGRAFICA_PLANO, COBERTURA_ASSIST_PLAN,
               TP_SEXO, DE_FAIXA_ETARIA, DE_FAIXA_ETARIA_REAJ, TIPO_VINCULO
  ORDER BY _ingested_at DESC
) = 1;

In [0]:
%sql
-- merge da siler.beneficiarios

MERGE INTO silver.beneficiarios_ans AS tgt
USING stg_silver AS src
ON  tgt.competencia    = src.competencia
AND tgt.cd_operadora   = src.cd_operadora
AND tgt.nm_municipio   = src.nm_municipio
AND tgt.cd_plano       = src.cd_plano
AND tgt.tp_vigencia_plano = src.tp_vigencia_plano
AND tgt.de_contratacao_plano = src.de_contratacao_plano
AND tgt.de_segmentacao_plano = src.de_segmentacao_plano
AND tgt.de_abrangencia_geografica_plano = src.de_abrangencia_geografica_plano
AND tgt.cobertura_assistencial_plano = src.cobertura_assistencial_plano
AND tgt.tp_sexo        = src.tp_sexo
AND tgt.de_faixa_etaria = src.de_faixa_etaria
AND tgt.de_faixa_etaria_reajuste = src.de_faixa_etaria_reajuste
AND tgt.tipo_vinculo   = src.tipo_vinculo
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql

-- compacta arquivo pequeno em arquivo grande + organiza fisicamente por essas colunas, pra query filtrada nelas ficar mais rápida.
OPTIMIZE silver.beneficiarios_ans ZORDER BY (cd_operadora, nm_municipio);

In [0]:
%sql
-- Checagem de qualidade
 
SELECT
  COUNT(*) AS total_linhas,
  SUM(CASE WHEN qt_beneficiario_ativo IS NULL THEN 1 ELSE 0 END) AS ativos_nulos,
  COUNT(DISTINCT competencia) AS competencias
FROM bmg_saude_desafio.silver.beneficiarios_ans

In [0]:
%sql
select * from workspace.